# 08. Construcción MILP de múltiples rondas

En los notebooks anteriores se implementaron y validaron individualmente las
transformaciones de una ronda de Keccak:

$$
\theta
\longrightarrow
\rho
\longrightarrow
\pi
\longrightarrow
\chi
\longrightarrow
\iota.
$$

La salida de `iota` de la ronda $r$ corresponde directamente al siguiente
estado de frontera:

$$
A_{r+1}.
$$

Por tanto, varias rondas consecutivas pueden representarse como:

$$
A_0
\longrightarrow
A_1
\longrightarrow
A_2
\longrightarrow
\cdots
\longrightarrow
A_R.
$$

Este notebook tiene los siguientes objetivos:

1. validar los métodos `add_round` y `add_all_rounds`;
2. estudiar el crecimiento del modelo según $z$ y el número de rondas;
3. resolver dos rondas consecutivas con CBC;
4. comparar todos los estados de frontera con la referencia;
5. validar las salidas internas de cada ronda;
6. repetir el experimento para diferentes configuraciones reproducibles.

In [1]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================================

from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Busca la raíz del proyecto a partir del directorio actual."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto con una carpeta `src`."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print("Raíz del proyecto:", PROJECT_ROOT)
print("Directorio src:", SRC_DIR)

Raíz del proyecto: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Directorio src: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src


In [2]:
# ============================================================
# IMPORTACIONES Y VALIDACIÓN DE LA API
# ============================================================

import inspect
import time

import numpy as np
from pulp import LpStatus

import keccak_milp.layers as layers

from keccak_milp.config import ExperimentConfig
from keccak_milp.model import KeccakMILPModel


required_layer_functions = {
    "keccak_round",
    "keccak_rounds",
    "theta",
    "rho_pi",
    "chi",
    "iota",
}

required_model_methods = {
    "add_round",
    "add_all_rounds",
    "theta_output_values",
    "rho_pi_output_values",
    "chi_output_values",
    "iota_output_values",
}

available_layer_functions = set(dir(layers))
available_model_methods = set(dir(KeccakMILPModel))

assert required_layer_functions.issubset(
    available_layer_functions
)

assert required_model_methods.issubset(
    available_model_methods
)


print("Funciones de referencia:")

for name in sorted(required_layer_functions):
    print(
        f" - {name}"
        f"{inspect.signature(getattr(layers, name))}"
    )


print("\nMétodos del modelo:")

for name in sorted(required_model_methods):
    print(
        f" - {name}"
        f"{inspect.signature(getattr(KeccakMILPModel, name))}"
    )


print("\nLa API de múltiples rondas está disponible.")

Funciones de referencia:
 - chi(state: 'NDArray[np.integer]') -> 'NDArray[np.int64]'
 - iota(state: 'NDArray[np.integer]', round_index: 'int') -> 'NDArray[np.int64]'
 - keccak_round(state: 'NDArray[np.integer]', round_index: 'int') -> 'NDArray[np.int64]'
 - keccak_rounds(state: 'NDArray[np.integer]', number_of_rounds: 'int', start_round: 'int' = 0) -> 'NDArray[np.int64]'
 - rho_pi(state: 'NDArray[T]') -> 'NDArray[T]'
 - theta(state: 'NDArray[np.integer]') -> 'NDArray[np.int64]'

Métodos del modelo:
 - add_all_rounds(self) -> 'None'
 - add_round(self, round_index: 'int') -> 'None'
 - chi_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'
 - iota_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'
 - rho_pi_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'
 - theta_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'

La API

## Crecimiento del modelo

Para una configuración con $R$ rondas se declaran $R+1$ estados de frontera.

Cada estado contiene:

$$
25z
$$

variables binarias.

Las variables internas creadas por una ronda son:

- `theta`: $70z$;
- `rho_pi`: $25z$;
- `chi`: $75z$;
- `iota`: $0$.

Por tanto, una ronda agrega:

$$
170z
$$

variables internas.

El número total de variables declaradas es:

$$
N_{\mathrm{variables}}
=
25z(R+1)+170zR.
$$

Simplificando:

$$
N_{\mathrm{variables}}
=
25z+195zR.
$$

Las restricciones agregadas por ronda son:

- `theta`: $35z$;
- `rho_pi`: $25z$;
- `chi`: $100z$;
- `iota`: $25z$.

Por tanto:

$$
N_{\mathrm{restricciones}}
=
185zR.
$$

El crecimiento del modelo es lineal respecto al número de rondas.

In [3]:
# ============================================================
# FUNCIONES PARA CALCULAR EL TAMAÑO ESPERADO
# ============================================================

def expected_declared_variables(
    z: int,
    rounds: int,
) -> int:
    """Calcula las variables declaradas esperadas."""
    return 25 * z + 195 * z * rounds


def expected_constraints(
    z: int,
    rounds: int,
) -> int:
    """Calcula las restricciones esperadas."""
    return 185 * z * rounds


configurations = [
    (4, 1),
    (4, 2),
    (4, 3),
    (8, 1),
    (8, 2),
    (8, 3),
]


print("z | Rondas | Variables | Restricciones")
print("-" * 42)

for z_value, round_count in configurations:
    print(
        f"{z_value:>1} | "
        f"{round_count:>6} | "
        f"{expected_declared_variables(z_value, round_count):>9} | "
        f"{expected_constraints(z_value, round_count):>13}"
    )

z | Rondas | Variables | Restricciones
------------------------------------------
4 |      1 |       880 |           740
4 |      2 |      1660 |          1480
4 |      3 |      2440 |          2220
8 |      1 |      1760 |          1480
8 |      2 |      3320 |          2960
8 |      3 |      4880 |          4440


In [4]:
# ============================================================
# CONSTRUCCIÓN AUTOMÁTICA DE DOS RONDAS
# ============================================================

z = 8
number_of_rounds = 2

structural_config = ExperimentConfig(
    z=z,
    rounds=number_of_rounds,
    solver="cbc",
    verbose=False,
)

multi_round_model = KeccakMILPModel(
    structural_config
)

multi_round_model.add_all_rounds()


declared_variables = (
    multi_round_model.declared_variable_count()
)

attached_variables = (
    multi_round_model.attached_variable_count()
)

total_constraints = (
    multi_round_model.constraint_count()
)


print("Modelo de dos rondas")
print("-" * 45)
print("Variables declaradas:", declared_variables)
print("Variables conectadas:", attached_variables)
print("Restricciones:", total_constraints)


assert declared_variables == expected_declared_variables(
    z,
    number_of_rounds,
)

assert total_constraints == expected_constraints(
    z,
    number_of_rounds,
)


print("\nEl modelo tiene el tamaño esperado.")

Modelo de dos rondas
---------------------------------------------
Variables declaradas: 3320
Variables conectadas: 3320
Restricciones: 2960

El modelo tiene el tamaño esperado.


In [5]:
# ============================================================
# IDEMPOTENCIA DE add_all_rounds
# ============================================================

variables_before = (
    multi_round_model.declared_variable_count()
)

constraints_before = (
    multi_round_model.constraint_count()
)


multi_round_model.add_all_rounds()


variables_after = (
    multi_round_model.declared_variable_count()
)

constraints_after = (
    multi_round_model.constraint_count()
)


assert variables_after == variables_before
assert constraints_after == constraints_before


print("Idempotencia verificada correctamente.")
print(
    "Variables antes y después:",
    variables_after,
)
print(
    "Restricciones antes y después:",
    constraints_after,
)

Idempotencia verificada correctamente.
Variables antes y después: 3320
Restricciones antes y después: 2960


In [6]:
# ============================================================
# ESTADO INICIAL CONTROLADO
# ============================================================

input_state = np.zeros(
    (5, 5, z),
    dtype=np.int64,
)

active_input_bits = [
    (0, 0, 0),
    (1, 0, 1),
    (0, 1, 3),
    (2, 3, 4),
    (3, 2, 5),
    (4, 4, 7),
]

for x, y, k in active_input_bits:
    input_state[x, y, k] = 1


print("Bits activos de A_0:")

for position in active_input_bits:
    print(" -", position)

print(
    "\nPeso de Hamming de A_0:",
    int(input_state.sum()),
)

Bits activos de A_0:
 - (0, 0, 0)
 - (1, 0, 1)
 - (0, 1, 3)
 - (2, 3, 4)
 - (3, 2, 5)
 - (4, 4, 7)

Peso de Hamming de A_0: 6


In [7]:
# ============================================================
# TRAZA DE REFERENCIA DE VARIAS RONDAS
# ============================================================

def build_reference_trace(
    initial_state: np.ndarray,
    rounds: int,
) -> list[dict[str, np.ndarray | int]]:
    """
    Calcula todos los estados y salidas internas de referencia.
    """
    current_state = np.asarray(
        initial_state,
        dtype=np.int64,
    ).copy()

    trace = []

    for round_index in range(rounds):
        theta_output = layers.theta(
            current_state.copy()
        )

        rho_pi_output = layers.rho_pi(
            theta_output.copy()
        )

        chi_output = layers.chi(
            rho_pi_output.copy()
        )

        next_state = layers.iota(
            chi_output.copy(),
            round_index=round_index,
        )

        trace.append(
            {
                "round_index": round_index,
                "input": current_state.copy(),
                "theta": theta_output,
                "rho_pi": rho_pi_output,
                "chi": chi_output,
                "iota": next_state,
            }
        )

        current_state = next_state

    return trace


reference_trace = build_reference_trace(
    input_state,
    number_of_rounds,
)


print(
    "Ronda | Entrada | Theta | Rho-Pi | Chi | Iota"
)
print("-" * 52)

for round_data in reference_trace:
    print(
        f"{round_data['round_index']:>5} | "
        f"{int(round_data['input'].sum()):>7} | "
        f"{int(round_data['theta'].sum()):>5} | "
        f"{int(round_data['rho_pi'].sum()):>6} | "
        f"{int(round_data['chi'].sum()):>3} | "
        f"{int(round_data['iota'].sum()):>4}"
    )

Ronda | Entrada | Theta | Rho-Pi | Chi | Iota
----------------------------------------------------
    0 |       6 |    66 |     66 |  89 |   88
    1 |      88 |    84 |     84 |  95 |   97


In [8]:
# ============================================================
# FUNCIONES AUXILIARES PARA RECUPERAR SOLUCIONES
# ============================================================

def normalize_solution_state(state) -> np.ndarray:
    """Convierte una salida del modelo a un arreglo binario."""
    array = np.asarray(
        state,
        dtype=float,
    )

    if np.isnan(array).any():
        raise RuntimeError(
            "La solución contiene valores no definidos."
        )

    return np.rint(array).astype(
        np.int64
    )


def boundary_state_values(
    model: KeccakMILPModel,
    boundary_index: int,
) -> np.ndarray:
    """Recupera un estado de frontera resuelto."""
    z_value = model.config.z

    output = np.zeros(
        (5, 5, z_value),
        dtype=np.int64,
    )

    for x in range(5):
        for y in range(5):
            for k in range(z_value):
                variable = model.state_variable(
                    round_index=boundary_index,
                    x=x,
                    y=y,
                    k=k,
                )

                value = variable.value()

                if value is None:
                    raise RuntimeError(
                        "El modelo debe resolverse antes de "
                        "recuperar un estado de frontera."
                    )

                output[x, y, k] = int(
                    value > 0.5
                )

    return output


def difference_count(
    first_state,
    second_state,
) -> int:
    """Cuenta las diferencias bit a bit."""
    first_array = np.asarray(
        first_state,
        dtype=np.int64,
    )

    second_array = np.asarray(
        second_state,
        dtype=np.int64,
    )

    return int(
        np.count_nonzero(
            first_array != second_array
        )
    )


print("Funciones auxiliares definidas correctamente.")

Funciones auxiliares definidas correctamente.


In [9]:
# ============================================================
# MODELO MILP DE DOS RONDAS CON ENTRADA FIJA
# ============================================================

validation_config = ExperimentConfig(
    z=z,
    rounds=number_of_rounds,
    solver="cbc",
    verbose=False,
)

validation_model = KeccakMILPModel(
    validation_config
)

validation_model.add_all_rounds()


# ------------------------------------------------------------
# Fijar A_0
# ------------------------------------------------------------

for x in range(5):
    for y in range(5):
        for k in range(z):
            variable = validation_model.state_variable(
                round_index=0,
                x=x,
                y=y,
                k=k,
            )

            validation_model.problem += (
                variable == int(input_state[x, y, k]),
                f"fix_a0_x{x}_y{y}_k{k}",
            )


validation_model.set_smoke_test_objective()


start_time = time.perf_counter()

solve_result = validation_model.solve()

elapsed_seconds = (
    time.perf_counter() - start_time
)


status_code = validation_model.problem.status

status_name = LpStatus.get(
    status_code,
    str(status_code),
)


print("Resultado:", solve_result)
print("Estado:", status_name)
print(
    "Tiempo de resolución:",
    f"{elapsed_seconds:.4f} segundos",
)
print(
    "Variables declaradas:",
    validation_model.declared_variable_count(),
)
print(
    "Restricciones:",
    validation_model.constraint_count(),
)


assert status_name == "Optimal"

Resultado: Optimal
Estado: Optimal
Tiempo de resolución: 0.3384 segundos
Variables declaradas: 3320
Restricciones: 3160


In [10]:
# ============================================================
# COMPARACIÓN DE LOS ESTADOS A_0, A_1 Y A_2
# ============================================================

milp_boundary_states = [
    boundary_state_values(
        validation_model,
        boundary_index,
    )
    for boundary_index in range(
        number_of_rounds + 1
    )
]

reference_boundary_states = [
    input_state,
    reference_trace[0]["iota"],
    reference_trace[1]["iota"],
]


print("Estado | Peso referencia | Peso MILP | Diferencias")
print("-" * 55)

for boundary_index in range(
    number_of_rounds + 1
):
    reference_state = (
        reference_boundary_states[
            boundary_index
        ]
    )

    milp_state = (
        milp_boundary_states[
            boundary_index
        ]
    )

    differences = difference_count(
        milp_state,
        reference_state,
    )

    print(
        f"A_{boundary_index:<4} | "
        f"{int(reference_state.sum()):>15} | "
        f"{int(milp_state.sum()):>9} | "
        f"{differences:>11}"
    )

    assert differences == 0


print(
    "\nTodos los estados de frontera coinciden "
    "con la referencia."
)

Estado | Peso referencia | Peso MILP | Diferencias
-------------------------------------------------------
A_0    |               6 |         6 |           0
A_1    |              88 |        88 |           0
A_2    |              97 |        97 |           0

Todos los estados de frontera coinciden con la referencia.


In [11]:
# ============================================================
# COMPARACIÓN DE LAS CAPAS INTERNAS DE CADA RONDA
# ============================================================

internal_results = []


for round_index in range(number_of_rounds):
    reference = reference_trace[
        round_index
    ]

    theta_milp = normalize_solution_state(
        validation_model.theta_output_values(
            round_index
        )
    )

    rho_pi_milp = normalize_solution_state(
        validation_model.rho_pi_output_values(
            round_index
        )
    )

    chi_milp = normalize_solution_state(
        validation_model.chi_output_values(
            round_index
        )
    )

    iota_milp = normalize_solution_state(
        validation_model.iota_output_values(
            round_index
        )
    )


    round_result = {
        "ronda": round_index,
        "theta": difference_count(
            theta_milp,
            reference["theta"],
        ),
        "rho_pi": difference_count(
            rho_pi_milp,
            reference["rho_pi"],
        ),
        "chi": difference_count(
            chi_milp,
            reference["chi"],
        ),
        "iota": difference_count(
            iota_milp,
            reference["iota"],
        ),
    }

    internal_results.append(
        round_result
    )


print(
    "Ronda | Theta | Rho-Pi | Chi | Iota"
)
print("-" * 38)

for result in internal_results:
    print(
        f"{result['ronda']:>5} | "
        f"{result['theta']:>5} | "
        f"{result['rho_pi']:>6} | "
        f"{result['chi']:>3} | "
        f"{result['iota']:>4}"
    )


assert all(
    result["theta"] == 0
    and result["rho_pi"] == 0
    and result["chi"] == 0
    and result["iota"] == 0
    for result in internal_results
)


print(
    "\nTodas las capas internas coinciden "
    "bit a bit con la referencia."
)

Ronda | Theta | Rho-Pi | Chi | Iota
--------------------------------------
    0 |     0 |      0 |   0 |    0
    1 |     0 |      0 |   0 |    0

Todas las capas internas coinciden bit a bit con la referencia.


## Interpretación de la propagación

La validación de los estados de frontera demuestra que la salida de `iota`
de una ronda se utiliza correctamente como entrada de la siguiente:

$$
A_{r+1}
=
\iota
\left(
\chi
\left(
\rho\pi
\left(
\theta(A_r)
\right)
\right),
r
\right).
$$

No es necesario copiar manualmente los bits entre rondas. La misma variable:

$$
\texttt{state}[r+1,x,y,k]
$$

cumple simultáneamente dos funciones:

- es la salida de `iota` de la ronda $r$;
- es la entrada de `theta` de la ronda $r+1$.

Esta reutilización reduce la cantidad de variables y evita restricciones de
copia adicionales entre rondas.

In [12]:
# ============================================================
# RESOLUCIÓN REUTILIZABLE DE VARIAS RONDAS
# ============================================================

def solve_multi_round_case(
    initial_state: np.ndarray,
    rounds: int,
) -> dict:
    """Resuelve varias rondas y compara todos los estados."""
    input_array = np.asarray(
        initial_state,
        dtype=np.int64,
    )

    if input_array.ndim != 3:
        raise ValueError(
            "El estado debe tener tres dimensiones."
        )

    if input_array.shape[:2] != (5, 5):
        raise ValueError(
            "Las dos primeras dimensiones deben ser 5 × 5."
        )

    local_z = input_array.shape[2]

    if local_z not in {4, 8}:
        raise ValueError(
            "El tamaño de palabra debe ser 4 u 8."
        )

    if rounds <= 0:
        raise ValueError(
            "El número de rondas debe ser positivo."
        )

    config = ExperimentConfig(
        z=local_z,
        rounds=rounds,
        solver="cbc",
        verbose=False,
    )

    model = KeccakMILPModel(config)
    model.add_all_rounds()

    for x in range(5):
        for y in range(5):
            for k in range(local_z):
                model.problem += (
                    model.state_variable(
                        0,
                        x,
                        y,
                        k,
                    )
                    == int(input_array[x, y, k]),
                    f"fix_multi_x{x}_y{y}_k{k}",
                )

    model.set_smoke_test_objective()

    start = time.perf_counter()
    status = model.solve()
    elapsed = time.perf_counter() - start

    if status != "Optimal":
        raise RuntimeError(
            f"CBC devolvió el estado {status}."
        )

    reference = layers.keccak_rounds(
        input_array,
        number_of_rounds=rounds,
    )

    obtained = boundary_state_values(
        model,
        rounds,
    )

    return {
        "z": local_z,
        "rondas": rounds,
        "estado_solver": status,
        "variables": (
            model.declared_variable_count()
        ),
        "restricciones": (
            model.constraint_count()
        ),
        "tiempo_segundos": elapsed,
        "peso_entrada": int(
            input_array.sum()
        ),
        "peso_salida": int(
            obtained.sum()
        ),
        "diferencias": difference_count(
            obtained,
            reference,
        ),
    }

In [13]:
# ============================================================
# VALIDACIÓN DE DISTINTAS CONFIGURACIONES
# ============================================================

experiment_cases = [
    {
        "seed": 7,
        "z": 4,
        "rounds": 2,
    },
    {
        "seed": 2026,
        "z": 8,
        "rounds": 2,
    },
    {
        "seed": 640,
        "z": 8,
        "rounds": 3,
    },
]

experiment_results = []


for case in experiment_cases:
    rng = np.random.default_rng(
        case["seed"]
    )

    random_input = rng.integers(
        low=0,
        high=2,
        size=(
            5,
            5,
            case["z"],
        ),
        dtype=np.int64,
    )

    result = solve_multi_round_case(
        initial_state=random_input,
        rounds=case["rounds"],
    )

    result["semilla"] = case["seed"]

    experiment_results.append(
        result
    )

    assert result["diferencias"] == 0


print(
    f"Se validaron correctamente "
    f"{len(experiment_results)} configuraciones."
)

Se validaron correctamente 3 configuraciones.


In [14]:
# ============================================================
# RESUMEN DE LOS EXPERIMENTOS
# ============================================================

header = (
    "z | Rondas | Variables | Restricciones | "
    "Tiempo (s) | Entrada | Salida | Diferencias"
)

print(header)
print("-" * len(header))


for result in experiment_results:
    print(
        f"{result['z']:>1} | "
        f"{result['rondas']:>6} | "
        f"{result['variables']:>9} | "
        f"{result['restricciones']:>13} | "
        f"{result['tiempo_segundos']:>10.4f} | "
        f"{result['peso_entrada']:>7} | "
        f"{result['peso_salida']:>6} | "
        f"{result['diferencias']:>11}"
    )

z | Rondas | Variables | Restricciones | Tiempo (s) | Entrada | Salida | Diferencias
------------------------------------------------------------------------------------
4 |      2 |      1660 |          1580 |     0.2734 |      56 |     42 |           0
8 |      2 |      3320 |          3160 |     0.3011 |      97 |    101 |           0
8 |      3 |      4880 |          4640 |     0.3654 |      90 |     96 |           0


In [15]:
# ============================================================
# COMPROBACIÓN GLOBAL
# ============================================================

assert all(
    result["estado_solver"] == "Optimal"
    for result in experiment_results
)

assert all(
    result["diferencias"] == 0
    for result in experiment_results
)

assert all(
    result["variables"]
    == expected_declared_variables(
        result["z"],
        result["rondas"],
    )
    for result in experiment_results
)

assert all(
    result["restricciones"]
    == (
        expected_constraints(
            result["z"],
            result["rondas"],
        )
        + 25 * result["z"]
    )
    for result in experiment_results
)


print("Todas las validaciones fueron superadas.")
print("-" * 45)
print(
    "Configuraciones evaluadas:",
    len(experiment_results),
)
print(
    "Errores encontrados:",
    sum(
        result["diferencias"] > 0
        for result in experiment_results
    ),
)

Todas las validaciones fueron superadas.
---------------------------------------------
Configuraciones evaluadas: 3
Errores encontrados: 0


## Conclusiones

La construcción de múltiples rondas consecutivas fue validada
estructural y funcionalmente.

Los principales resultados son:

1. El método `add_round(r)` agrega automáticamente:

   $$
   \theta
   \longrightarrow
   \rho
   \longrightarrow
   \pi
   \longrightarrow
   \chi
   \longrightarrow
   \iota.
   $$

2. El método `add_all_rounds()` construye todas las rondas configuradas en
   el orden correcto.

3. Ambos métodos son compatibles con las operaciones idempotentes de las
   capas individuales.

4. El número de variables declaradas para $R$ rondas es:

   $$
   N_{\mathrm{variables}}
   =
   25z+195zR.
   $$

5. El número de restricciones de las rondas es:

   $$
   N_{\mathrm{restricciones}}
   =
   185zR.
   $$

6. La salida de `iota` de la ronda $r$ es la misma variable que representa
   la entrada de la ronda $r+1$.

7. No se necesitan variables ni restricciones adicionales para copiar
   estados entre rondas.

8. Los estados de frontera:

   $$
   A_0,A_1,\ldots,A_R
   $$

   coinciden bit a bit con la implementación de referencia.

9. Las salidas internas de `theta`, `rho_pi`, `chi` e `iota` también
   coinciden en cada ronda.

10. La validación se realizó para $z=4$, $z=8$, dos rondas y tres rondas.

Con esta etapa, el proyecto dispone de una representación MILP funcional
de múltiples rondas reducidas de Keccak:

$$
A_0
\longrightarrow
A_1
\longrightarrow
\cdots
\longrightarrow
A_R.
$$

La siguiente etapa será reemplazar el objetivo provisional por una función
objetivo criptanalítica orientada a minimizar el peso activo de una
diferencia o trayectoria.